# Notebook 01: LLM Model Selection and Setup — OFPHIL MSW Interview Assistant

**Project:** `ofphil-classification-v2`  
**Phase:** 05 Model Selection and Training — Track B: LLM Interview Assistant  
**Notebook:** `01_llm_model_selection_and_setup.ipynb`  
**Role:** Stage 2 Medical Social Worker (MSW) Interview Assistant (Conversational Guidance & Structured Extraction)

---

### Objectives
1. Define the functional requirements and architectural boundaries for the OFPHIL MSW Interview Assistant.
2. Formulate an evidence-based multi-criteria evaluation framework (Hardware, Latency, Privacy, Language, Cost, Structured Output).
3. Evaluate candidate LLM families across **Local Edge Models** (e.g., Llama 3.1 8B, Mistral 7B, Qwen 2.5 7B) and **API Cloud Services** (e.g., Gemini 1.5 Pro/Flash, Claude 3.5 Sonnet, GPT-4o-mini).
4. Evaluate integration compatibility with the existing FastAPI Python 3.12 backend.
5. Document selection trade-offs, operational risks, and institutional data privacy protocols without making external API calls or downloading large weights.
6. Export candidate model specifications and architecture configuration to `data/interim/llm_interview/model_selection_matrix.csv` and `data/interim/llm_interview/llm_configuration.json`.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

def _find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'data').exists() and (p / 'experiments').exists():
            return p
    return Path('../../..').resolve()

PROJECT_ROOT = _find_project_root(Path.cwd())
LLM_INTERIM_DIR = PROJECT_ROOT / 'data' / 'interim' / 'llm_interview'
CONFIG_DIR = LLM_INTERIM_DIR / 'config'
LLM_INTERIM_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

MODEL_MATRIX_CSV = LLM_INTERIM_DIR / 'model_selection_matrix.csv'
CONFIG_JSON_OUT = CONFIG_DIR / 'llm_configuration.json'

print(f"Project root: {PROJECT_ROOT}")
print(f"LLM Interim Directory: {LLM_INTERIM_DIR}")


### 2. Multi-Criteria Model Evaluation Framework
The MSW Interview Assistant serves as an interactive decision-support copilot during Stage 2 intake. The model must satisfy six strict institutional criteria:
1. **Privacy & Data Sovereignty:** Patient interview statements contain sensitive psychosocial facts. Cloud APIs require BAA/Data Processing Agreements, while local models ensure strict on-premise containment.
2. **Filipino / Taglish Linguistic Competency:** Low-income charity patients frequently communicate in Tagalog or Taglish colloquialisms.
3. **Structured JSON Output Support:** The model must support schema-constrained decoding or grammar enforcement (JSON mode).
4. **Inference Latency:** MSW conversational guidance requires sub-3-second responses to maintain interview flow.
5. **Hardware Footprint:** Local models must fit within available clinic workstations (e.g., NVIDIA RTX 4080/4090 or Apple Silicon / CPU quantization).
6. **Cost & Operational Simplicity:** Long-term sustainability without excessive recurring API expenses.


In [ ]:
# Define structured candidate model evaluation matrix
candidates = [
    {
        "candidate_id": "LLM-CAND-01",
        "model_name": "Llama-3.1-8B-Instruct (4-bit / 8-bit Local)",
        "deployment_type": "Local / On-Premise (Ollama / vLLM / llama.cpp)",
        "parameter_size": "8.0B",
        "context_window_tokens": 131072,
        "hardware_requirements": "1x NVIDIA RTX 3060/4060 (8GB-12GB VRAM) or Apple M-series",
        "tagalog_taglish_competency": "Moderate to High (strong base multilingual tokenization)",
        "structured_json_mode": "Native (via GBNF grammars / Outlines / Ollama JSON mode)",
        "privacy_and_hipaa_dpa": "Highest (Zero external egress; 100% on-premise data residency)",
        "est_latency_sec": 1.8,
        "api_cost_per_1k_tokens": 0.00,
        "license": "Llama 3.1 Community License (Permissive Commercial)",
        "fastapi_integration": "Direct via local REST endpoint (Ollama client / httpx async client)"
    },
    {
        "candidate_id": "LLM-CAND-02",
        "model_name": "Qwen-2.5-7B-Instruct (4-bit Local)",
        "deployment_type": "Local / On-Premise (vLLM / Ollama)",
        "parameter_size": "7.6B",
        "context_window_tokens": 131072,
        "hardware_requirements": "1x NVIDIA GPU with >= 8GB VRAM",
        "tagalog_taglish_competency": "High (exceptional multilingual instruction adherence)",
        "structured_json_mode": "Native (Grammar-guided decoding)",
        "privacy_and_hipaa_dpa": "Highest (100% On-premise)",
        "est_latency_sec": 1.5,
        "api_cost_per_1k_tokens": 0.00,
        "license": "Apache 2.0 (Fully open-source)",
        "fastapi_integration": "Direct via local REST endpoint (vLLM / Ollama)"
    },
    {
        "candidate_id": "LLM-CAND-03",
        "model_name": "Mistral-7B-Instruct-v0.3 (Local)",
        "deployment_type": "Local / On-Premise (llama.cpp)",
        "parameter_size": "7.3B",
        "context_window_tokens": 32768,
        "hardware_requirements": "1x GPU >= 8GB VRAM or 16GB System RAM (CPU/GGUF)",
        "tagalog_taglish_competency": "Moderate (limited informal Tagalog phrasing understanding)",
        "structured_json_mode": "Native Function Calling / JSON mode",
        "privacy_and_hipaa_dpa": "Highest (On-premise)",
        "est_latency_sec": 2.1,
        "api_cost_per_1k_tokens": 0.00,
        "license": "Apache 2.0",
        "fastapi_integration": "Direct via llama-cpp-python / Ollama"
    },
    {
        "candidate_id": "LLM-CAND-04",
        "model_name": "Gemini 1.5 Flash (Cloud API)",
        "deployment_type": "Cloud API (Google AI Studio / Vertex AI)",
        "parameter_size": "Proprietary (~Efficient MoE)",
        "context_window_tokens": 1048576,
        "hardware_requirements": "Zero local GPU requirement (Standard HTTP server)",
        "tagalog_taglish_competency": "Very High (outstanding multilingual & regional nuance handling)",
        "structured_json_mode": "Native (Pydantic / Structured Outputs SDK schema enforcement)",
        "privacy_and_hipaa_dpa": "Enterprise DPA required (Zero training on customer prompts)",
        "est_latency_sec": 0.8,
        "api_cost_per_1k_tokens": 0.0001,
        "license": "Commercial API SLA",
        "fastapi_integration": "google-genai / google-generativeai SDK with async client"
    },
    {
        "candidate_id": "LLM-CAND-05",
        "model_name": "GPT-4o-mini (Cloud API)",
        "deployment_type": "Cloud API (OpenAI)",
        "parameter_size": "Proprietary",
        "context_window_tokens": 128000,
        "hardware_requirements": "Zero local GPU requirement",
        "tagalog_taglish_competency": "High",
        "structured_json_mode": "Native Structured Outputs (Guaranteed strict schema compliance)",
        "privacy_and_hipaa_dpa": "Enterprise Zero Data Retention agreement required",
        "est_latency_sec": 1.1,
        "api_cost_per_1k_tokens": 0.00015,
        "license": "Commercial API SLA",
        "fastapi_integration": "openai SDK / httpx async client"
    }
]

model_matrix_df = pd.DataFrame(candidates)
model_matrix_df.to_csv(MODEL_MATRIX_CSV, index=False)
print(f"Generated candidate model selection matrix ({len(model_matrix_df)} candidates):")
print(model_matrix_df[['candidate_id', 'model_name', 'deployment_type', 'privacy_and_hipaa_dpa', 'fastapi_integration']].to_string(index=False))


### 3. Tiered Hybrid Architecture Recommendation

Based on healthcare data governance and clinical operability requirements:
- **Primary Operational Recommendation (Hospital Clinic Deployment):** **Qwen 2.5 7B Instruct** or **Llama 3.1 8B Instruct** deployed locally via **Ollama / vLLM**. This provides complete zero-egress patient privacy, deterministic JSON grammar decoding, and zero per-token financial costs.
- **Development & Research Baseline:** **Gemini 1.5 Flash / Cloud API** with strictly anonymized/synthetic prompt inputs, leveraging structured output schemas and sub-second latency for rapid prompt prototyping and benchmarking.
- **Architectural Abstraction:** The FastAPI backend implements an agnostic `LLMProvider` interface (via standard `AsyncOpenAI` or local HTTP client) allowing seamless switching between local Ollama endpoints and cloud APIs without altering prompt engineering logic.


In [ ]:
# Define standardized LLM client configuration and architectural parameters
llm_runtime_config = {
    "system_name": "OFPHIL MSW Interview Assistant Runtime",
    "version": "1.0.0",
    "target_phase": "Phase 05 — Track B: LLM Interview Assistant",
    "operational_mode": "Hybrid-Agnostic (Local Ollama default with Cloud API fallback)",
    "local_primary_candidate": {
        "candidate_id": "LLM-CAND-02",
        "model_identifier": "qwen2.5:7b-instruct-q4_K_M",
        "runtime_engine": "Ollama / vLLM",
        "context_limit": 32768,
        "temperature": 0.2,
        "top_p": 0.9,
        "max_tokens": 1024,
        "structured_format": "json"
    },
    "cloud_baseline_candidate": {
        "candidate_id": "LLM-CAND-04",
        "model_identifier": "gemini-1.5-flash",
        "runtime_engine": "Google Vertex AI / Google GenAI SDK",
        "temperature": 0.1,
        "max_tokens": 1024,
        "structured_format": "json"
    },
    "guardrail_parameters": {
        "enforce_zero_eligibility_determination": True,
        "enforce_zero_severity_scoring": True,
        "require_msw_confirmation_for_all_extractions": True,
        "strict_phi_masking": True,
        "rag_vector_db_enabled": False
    }
}

CONFIG_JSON_OUT.write_text(json.dumps(llm_runtime_config, indent=2), encoding='utf-8')
print(f"Exported LLM runtime configuration -> {CONFIG_JSON_OUT}")


### Summary of Notebook 01 Deliverables
- [x] Defined the 6-dimension evaluation framework for the MSW Interview Assistant.
- [x] Cataloged 5 leading candidate architectures (3 local edge, 2 cloud API).
- [x] Established the hybrid-agnostic architectural recommendation.
- [x] Exported `model_selection_matrix.csv` and `llm_configuration.json` without external network dependencies.
